# 06 — Cross-domain correlations & LNM–VLSM overlap


In [ ]:
import sys; sys.path.insert(0, "utils")
from config import *
from plotting import plot_combined_multimodal_figure, calculate_similarity_matrices

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from nilearn import plotting

import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

# Manuscript pretty names for the PALM variant families.
FAMILY_PRETTY = {
    FAM_LNM_NOCOV: "LNM — no volume covariate",
    FAM_LNM_NIHSS: "LNM — volume + NIHSS",
    FAM_VLSM:      "VLSM",
}

print("PROJECT_ROOT :", PROJECT_ROOT)
print("PALM_DIR     :", PALM_DIR)
print("FIG_DIR      :", FIG_DIR)
print("THR (-log10 0.05):", round(float(THR), 4))
print()
print("CORE_FAMILY:", CORE_FAMILY, "->", FAMILY_PRETTY[CORE_FAMILY])
print("SUPPLEMENT_FAMILIES:", SUPPLEMENT_FAMILIES)
print("LNM_FAMILIES:", LNM_FAMILIES)

## Shared loaders & gating helpers

In [ ]:
brain, affine, bg_path = load_mask()
bg_img = nib.load(bg_path)
N_brain = int(brain.sum())

# Display ordering / pretty names: degree first, then the six domains (matches reference figures).
label_rename_dict = {"degree": "Degree", **DOMAINS}

def have(family, key, fname):
    """True if a specific PALM output file exists for one variant."""
    return result_path(family, key, fname).exists()

def ready_keys(family, fname):
    """Domains for which <fname> exists under <family>."""
    return [k for k in KEYS if have(family, key=k, fname=fname)]

def sig_mask(family, key, pmap):
    """Boolean significant mask: in-brain, -log10(p) >= THR."""
    d = load_map(family, key, pmap, brain=brain)   # out-of-brain -> 0.0
    return (d >= THR) & brain

def signed_t(family, key):
    """In-brain signed T map; out-of-brain set to NaN (matches reference correlation code)."""
    arr = nib.load(result_path(family, key, F_TSTAT)).get_fdata()
    return np.where(brain, arr, np.nan)

# --- connectome degree (continuous + 95th-percentile binary) ---
degree = nib.load(DEGREE_MAP).get_fdata()
_dfin = degree[np.isfinite(degree)]
degree_thr = np.percentile(_dfin, 95)
degree_bin = degree > degree_thr
print("degree map:", DEGREE_MAP.name, "| 95th pct threshold =", round(float(degree_thr), 4))

## Cross-domain specificity + degree alignment

In [ ]:
def build_dicts(family, pmap, keys):
    """(binary_dict, continuous_dict) with the degree entry appended."""
    bd = {k: sig_mask(family, k, pmap) for k in keys}
    cd = {k: signed_t(family, k) for k in keys}
    bd["degree"] = degree_bin
    cd["degree"] = degree
    return bd, cd

def build_crossdomain_figure(family, pmap, scheme_label, out_name):
    """Build + save the combined multimodal figure for one family/scheme. Returns the fig or None."""
    rk = ready_keys(family, pmap)
    if len(rk) < 2:
        print(f"[skip] {FAMILY_PRETTY[family]} ({scheme_label}): only {len(rk)}/{len(KEYS)} "
              "domains ready -- skipping cross-domain figure.")
        return None
    if len(rk) < len(KEYS):
        print(f"Note: only {len(rk)}/{len(KEYS)} domains ready for {family} ({scheme_label}). "
              "Building matrix on the ready subset.")
    bd, cd = build_dicts(family, pmap, rk)
    fig = plot_combined_multimodal_figure(
        binary_dict=bd, continuous_dict=cd, affine=affine, bg_img=bg_img,
        label_rename_dict=label_rename_dict, fontsize=14, figsize_base=3,
        dot_size_factor=1600, label_wrap_width=20,
    )
    out = FIG_DIR / out_name
    fig.savefig(out, dpi=300, bbox_inches="tight")
    print("saved", out)
    return fig

def summarise_scheme(binary_dict, continuous_dict):
    """Mean off-diagonal cross-domain Dice & Pearson-r (domains only; degree excluded),
    plus mean degree-alignment Dice/r. Returns a dict of summary scalars."""
    dom_keys = [k for k in binary_dict if k != "degree"]
    keys_all = dom_keys + (["degree"] if "degree" in binary_dict else [])
    dice_df, pear_df = calculate_similarity_matrices(binary_dict, continuous_dict, keys=keys_all)
    dsub = dice_df.loc[dom_keys, dom_keys].values
    psub = pear_df.loc[dom_keys, dom_keys].values
    iu = np.triu_indices_from(dsub, k=1)
    out = {
        "n_domains": len(dom_keys),
        "dice_mean": float(np.nanmean(dsub[iu])),
        "dice_sd":   float(np.nanstd(dsub[iu])),
        "pearson_mean": float(np.nanmean(psub[iu])),
        "pearson_sd":   float(np.nanstd(psub[iu])),
    }
    if "degree" in binary_dict:
        out["degree_dice_mean"]    = float(np.nanmean(dice_df.loc[dom_keys, "degree"].values))
        out["degree_pearson_mean"] = float(np.nanmean(pear_df.loc[dom_keys, "degree"].values))
    return out

# Per-family scheme list: lnm_nocov reports both schemes; the other families report PERMUTATION ONLY
# (parametric figures/rows for those families are intentionally dropped).
def family_schemes(family):
    if family == CORE_FAMILY:
        return [(F_PERM_FDR, "Permutation LNM (TFCE FDR)"),
                (F_PARAM_FDR, "Parametric LNM (vox FDR)")]
    return [(F_PERM_FDR, "Permutation LNM (TFCE FDR)")]

def summarise_family(family):
    """Summary rows for one family (gated independently); schemes per family_schemes()."""
    rows = []
    for pmap, scheme in family_schemes(family):
        rk = ready_keys(family, pmap)
        if len(rk) >= 2:
            bd, cd = build_dicts(family, pmap, rk)
            rows.append({"family": family, "family_pretty": FAMILY_PRETTY[family],
                         "scheme": scheme, **summarise_scheme(bd, cd)})
    return rows

## Cross-domain specificity — `lnm_nocov`

In [ ]:
# ===== lnm_nocov, permutation =====
fig_main_perm = build_crossdomain_figure(
    CORE_FAMILY, F_PERM_FDR, "permutation",
    f"crossdomain_multimodal__{CORE_FAMILY}__perm.png")
plt.show()

In [ ]:
fig_main_param = build_crossdomain_figure(
    CORE_FAMILY, F_PARAM_FDR, "parametric",
    f"crossdomain_multimodal__{CORE_FAMILY}__param.png")
plt.show()

In [ ]:
main_rows = summarise_family(CORE_FAMILY)

## Cross-domain specificity — `lnm_volnihss` (volume + NIHSS)

In [ ]:
SUPP_LNM = [f for f in LNM_FAMILIES if f != CORE_FAMILY]   # volnihss
supp_figs = {}
for fam in SUPP_LNM:
    print(f"--- {FAMILY_PRETTY[fam]} ({fam}) ---")
    supp_figs[(fam, "perm")] = build_crossdomain_figure(
        fam, F_PERM_FDR, "permutation", f"crossdomain_multimodal__{fam}__perm.png")
    plt.show()

### One-sample t-test

In [ ]:
fig_onesample = build_crossdomain_figure(
    FAM_ONESAMPLE, F_PARAM_FDR, "parametric (one-sample t-test)",
    "crossdomain_multimodal__onesample__param.png")
plt.show()

## Summay across families

In [ ]:
all_rows = []
for fam in LNM_FAMILIES:                 # nocov first, then volnihss
    all_rows.extend(summarise_family(fam))

if not all_rows:
    print("PALM not finished -- no cross-domain summary available yet.")
else:
    combined = pd.DataFrame(all_rows)
    cols = ["family", "family_pretty", "scheme", "n_domains",
            "dice_mean", "dice_sd", "pearson_mean", "pearson_sd",
            "degree_dice_mean", "degree_pearson_mean"]
    combined = combined[[c for c in cols if c in combined.columns]].round(3)
    print(combined.to_string(index=False))
    out = FIG_DIR / "crossdomain_summary_allfamilies.csv"
    combined.to_csv(out, index=False)

## 3. Overlap figures — core family `lnm_nocov` only

In [ ]:
def plot_binary_comparison(binary_dict1, binary_dict2, affine, label_rename_dict=None,
                           figsize_per_col=6, name1="Map 1", name2="Map 2",
                           color1='#CC3333', color2='#0077BB',
                           fontsize=14, bg_img=None):
    """Side-by-side per-domain overlay of two binary-map dicts (red vs blue, overlap purple).

    Styling: NO family pretty-name title; instead a small two-colour key naming only the two
    maps (<name1> red, <name2> blue). Domains with NO surviving voxels in EITHER mask are drawn
    as an 'n.s.' text label (not a black/empty brain slice). Per-domain Dice for non-empty facets.

    Returns (fig, axes, dice_scores, common_keys) with dice_scores aligned to sorted common keys."""
    common_keys = sorted(set(binary_dict1) & set(binary_dict2))
    n = len(common_keys)
    if n == 0:
        raise ValueError("No matching keys found in input dictionaries.")
    print(f"Found {n} matching comparisons: {common_keys}")

    def get_label(k):
        return label_rename_dict.get(k, k) if label_rename_dict else k

    n_rows = 2
    n_cols = int(np.ceil(n / n_rows))
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(figsize_per_col * n_cols, figsize_per_col * n_rows))
    plt.subplots_adjust(wspace=0.0, hspace=0.2)
    axes_flat = axes.flatten() if n > 1 else [axes]
    dice_scores = np.zeros(n)

    for idx, key in enumerate(common_keys):
        ax = axes_flat[idx]
        mask1 = binary_dict1[key].astype(bool)
        mask2 = binary_dict2[key].astype(bool)
        img1 = nib.Nifti1Image(mask1.astype(np.float32), affine)
        img2 = nib.Nifti1Image(mask2.astype(np.float32), affine)
        cmap_i = ListedColormap([color1])  # dict1
        cmap_j = ListedColormap([color2])  # dict2

        if mask1.any() or mask2.any():
            if mask1.any():
                display = plotting.plot_stat_map(
                    img1, bg_img=bg_img, axes=ax, display_mode='z', cut_coords=[2],
                    cmap=cmap_i, colorbar=False, annotate=False, draw_cross=False,
                    threshold=0.1, transparency=0.6, dim=0.8, black_bg=False, radiological=True)
                if mask2.any():
                    display.add_overlay(img2, cmap=cmap_j, transparency=0.6)
            else:
                plotting.plot_stat_map(
                    img2, bg_img=bg_img, axes=ax, display_mode='z', cut_coords=[2],
                    cmap=cmap_j, colorbar=False, annotate=False, draw_cross=False,
                    threshold=0.1, transparency=0.6, dim=0.8, black_bg=False, radiological=True)
            ax.text(0.02, 0.55, 'R', transform=ax.transAxes, fontsize=10,
                    fontweight='bold', color='white')
            ax.text(0.98, 0.55, 'L', transform=ax.transAxes, fontsize=10,
                    fontweight='bold', color='white', ha='right')
        else:
            # No surviving voxels in EITHER mask -> label the facet n.s. (not a black slice).
            ax.set_facecolor('white')
            ax.text(0.5, 0.5, 'n.s.', transform=ax.transAxes, ha='center', va='center',
                    fontsize=fontsize * 1.4, fontweight='bold', color='0.4')
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

        ax.set_title(get_label(key), fontsize=fontsize, fontweight='bold', pad=20)

        inter = (mask1 & mask2).sum()
        tot = mask1.sum() + mask2.sum()
        d = 2.0 * inter / tot if tot > 0 else 0.0
        dice_scores[idx] = d
        if d > 0:
            ax.text(0.9, 0.05, f"Dice\n{d:.2f}", transform=ax.transAxes,
                    ha='center', va='center', fontsize=fontsize,
                    fontweight='bold', color='black')

    for i in range(n, len(axes_flat)):
        axes_flat[i].axis('off')

    # Small two-colour KEY naming only the two maps (replaces the family pretty-name title).
    legend_handles = [Patch(facecolor=color1, edgecolor='none', label=name1),
                      Patch(facecolor=color2, edgecolor='none', label=name2)]
    fig.legend(handles=legend_handles, loc='lower center', ncol=2, frameon=False,
               fontsize=fontsize, bbox_to_anchor=(0.5, 0.02))
    return fig, axes, dice_scores, common_keys

In [ ]:
def overlap_figure(binary_dict1, binary_dict2, out_name, name1, name2,
                   color1='#CC3333', color2='#0077BB'):
    """Build + save one core-family overlap figure. Returns (dice_scores, common_keys)."""
    fig, axes, dice_scores, common_keys = plot_binary_comparison(
        binary_dict1=binary_dict1, binary_dict2=binary_dict2, affine=affine,
        label_rename_dict=label_rename_dict, name1=name1, name2=name2,
        color1=color1, color2=color2, fontsize=20, bg_img=bg_img,
    )
    out_fig = FIG_DIR / out_name
    fig.savefig(out_fig, dpi=300, bbox_inches="tight")
    print("  saved", out_fig)
    plt.show()
    return dice_scores, common_keys

### 3a. Permutation vs parametric overlap (`lnm_nocov`)

In [ ]:
family = CORE_FAMILY
rk_perm  = ready_keys(family, F_PERM_FDR)
rk_param = ready_keys(family, F_PARAM_FDR)
shared   = [k for k in KEYS if k in rk_perm and k in rk_param]
print(f"[{family}] {FAMILY_PRETTY[family]} ({family}) -- permutation vs parametric")
if len(shared) == 0:
    print(f"  PALM not finished (perm {len(rk_perm)}/{len(KEYS)}, param {len(rk_param)}/{len(KEYS)}) -- skipping.")
    pvp_main = None
else:
    if len(shared) < len(KEYS):
        print(f"  Note: comparing the {len(shared)} domains ready under BOTH schemes.")
    perm_sig  = {k: sig_mask(family, k, F_PERM_FDR)  for k in shared}
    param_sig = {k: sig_mask(family, k, F_PARAM_FDR) for k in shared}
    dice_scores, common_keys = overlap_figure(
        perm_sig, param_sig, f"overlap__{family}__perm_vs_param.png",
        name1="Permutation", name2="Parametric")
    pvp_main = pd.DataFrame({
        "family": family, "family_pretty": FAMILY_PRETTY[family],
        "comparison": "perm_vs_param",
        "domain": [DOMAINS[k] for k in common_keys],
        "perm_voxels":  [int(perm_sig[k].sum())  for k in common_keys],
        "param_voxels": [int(param_sig[k].sum()) for k in common_keys],
        "dice_perm_param": np.round(dice_scores, 3),
    })
    print(pvp_main.drop(columns=["family", "family_pretty", "comparison"]).to_string(index=False))
    nz = pvp_main[(pvp_main.perm_voxels > 0) & (pvp_main.param_voxels > 0)]["dice_perm_param"]
    if len(nz):
        print(f"  Mean perm-vs-param Dice (non-empty domains): {nz.mean():.2f} +/- {nz.std():.2f}  (n={len(nz)})")
    out_csv = FIG_DIR / f"overlap_dice__{family}__perm_vs_param.csv"
    pvp_main.to_csv(out_csv, index=False)
    print("  saved", out_csv)

### 3b. Permutation group vs one-sample overlap (`lnm_nocov`)

In [ ]:
# ===== overlap: lnm_nocov group (permutation) vs one-sample (parametric) =====
family = CORE_FAMILY
rk_grp = ready_keys(family,        F_PERM_FDR)
rk_one = ready_keys(FAM_ONESAMPLE, F_PARAM_FDR)
shared = [k for k in KEYS if k in rk_grp and k in rk_one]
print(f"[{family}] {FAMILY_PRETTY[family]} ({family}) vs one-sample ({FAM_ONESAMPLE}) -- group permutation vs one-sample parametric")
if len(shared) == 0:
    print(f"  PALM not finished (group {len(rk_grp)}/{len(KEYS)}, onesample {len(rk_one)}/{len(KEYS)}) -- skipping.")
    dice_grp_one = None
else:
    if len(shared) < len(KEYS):
        print(f"  Note: comparing the {len(shared)} domains ready under BOTH analyses.")
    grp_sig = {k: sig_mask(family,        k, F_PERM_FDR) for k in shared}
    one_sig = {k: sig_mask(FAM_ONESAMPLE, k, F_PARAM_FDR) for k in shared}
    dice_scores, common_keys = overlap_figure(
        grp_sig, one_sig, f"overlap__{family}__perm_vs_onesample.png",
        name1="Permutation", name2="One-sample")
    dice_grp_one = pd.DataFrame({
        "family": family, "comparison": "perm_group_vs_onesample",
        "domain": [DOMAINS[k] for k in common_keys],
        "group_voxels":     [int(grp_sig[k].sum()) for k in common_keys],
        "onesample_voxels": [int(one_sig[k].sum()) for k in common_keys],
        "dice_group_onesample": np.round(dice_scores, 3),
    })
    print(dice_grp_one.drop(columns=["family", "comparison"]).to_string(index=False))
    nz = dice_grp_one[(dice_grp_one.group_voxels > 0) & (dice_grp_one.onesample_voxels > 0)]["dice_group_onesample"]
    if len(nz):
        print(f"  Mean group-vs-onesample Dice (non-empty domains): {nz.mean():.2f} +/- {nz.std():.2f}  (n={len(nz)})")
    out_csv = FIG_DIR / f"overlap_dice__{family}__perm_vs_onesample.csv"
    dice_grp_one.to_csv(out_csv, index=False)
    print("  saved", out_csv)

### 3c. LNM–VLSM overlap (`lnm_nocov` vs `vlsm_volcov`)

In [ ]:
# ===== overlap: lnm_nocov (LNM, permutation) vs vlsm_volcov (VLSM, permutation) =====
family = CORE_FAMILY
rk_lnm  = ready_keys(family,   F_PERM_FDR)
rk_vlsm = ready_keys(FAM_VLSM, F_PERM_FDR)
shared  = [k for k in KEYS if k in rk_lnm and k in rk_vlsm]
print(f"[{family}] {FAMILY_PRETTY[family]} ({family}) vs {FAMILY_PRETTY[FAM_VLSM]} ({FAM_VLSM}) -- permutation")
if len(shared) == 0:
    print(f"  PALM not finished (LNM {len(rk_lnm)}/{len(KEYS)}, VLSM {len(rk_vlsm)}/{len(KEYS)}) -- skipping.")
    dice_main = None
else:
    if len(shared) < len(KEYS):
        print(f"  Note: comparing the {len(shared)} domains ready under BOTH families.")
    lnm_sig  = {k: sig_mask(family,   k, F_PERM_FDR) for k in shared}
    vlsm_sig = {k: sig_mask(FAM_VLSM, k, F_PERM_FDR) for k in shared}
    dice_scores, common_keys = overlap_figure(
        lnm_sig, vlsm_sig, f"lnm_vlsm_overlap__{family}_vs_{FAM_VLSM}.png",
        name1="LNM", name2="VLSM")
    dice_main = pd.DataFrame({
        "family": family, "family_pretty": FAMILY_PRETTY[family],
        "domain": [DOMAINS[k] for k in common_keys],
        "lnm_voxels":  [int(lnm_sig[k].sum())  for k in common_keys],
        "vlsm_voxels": [int(vlsm_sig[k].sum()) for k in common_keys],
        "dice_lnm_vlsm": np.round(dice_scores, 3),
    })
    print(dice_main.drop(columns=["family", "family_pretty"]).to_string(index=False))
    nz = dice_main[(dice_main.lnm_voxels > 0) & (dice_main.vlsm_voxels > 0)]["dice_lnm_vlsm"]
    if len(nz):
        print(f"  Mean LNM-VLSM Dice (non-empty domains): {nz.mean():.2f} +/- {nz.std():.2f}  (n={len(nz)})")
    out_csv = FIG_DIR / f"lnm_vlsm_dice_table__{family}_vs_{FAM_VLSM}.csv"
    dice_main.to_csv(out_csv, index=False)
    print("  saved", out_csv)

## 4. Per-analysis spatial-correlation / Dice summary (per analysis family)

In [ ]:
def family_scheme_spatial_metrics(family, pmap):
    """Per (family, scheme) spatial-similarity metrics over the six domains, using the SAME
    degree_bin / degree maps and in-brain masking as the cross-domain multimodal figures.
    Returns a dict of metric -> (mean, std, n), or None if <2 domains are ready."""
    rk = ready_keys(family, pmap)
    if len(rk) < 2:
        return None
    bd, cd = build_dicts(family, pmap, rk)            # appends degree_bin / degree
    dom_keys = list(rk)                               # domain keys (degree handled separately)
    keys_all = dom_keys + ["degree"]
    dice_df, pear_df = calculate_similarity_matrices(bd, cd, keys=keys_all)

    dsub = dice_df.loc[dom_keys, dom_keys].values
    psub = pear_df.loc[dom_keys, dom_keys].values
    iu = np.triu_indices_from(dsub, k=1)
    xdom_dice = dsub[iu]
    xdom_pear = psub[iu]

    deg_dice = dice_df.loc[dom_keys, "degree"].values
    deg_pear = pear_df.loc[dom_keys, "degree"].values

    def ms(a):
        a = np.asarray(a, dtype=float)
        a = a[np.isfinite(a)]
        return float(np.mean(a)), float(np.std(a)), int(a.size)

    return {
        "cross_domain_dice":        ms(xdom_dice),
        "cross_domain_correlation": ms(xdom_pear),
        "degree_dice":              ms(deg_dice),
        "degree_correlation":       ms(deg_pear),
    }

# Per-family scheme list for the spatial summary: lnm_nocov = both schemes; others = perm only.
def summary_schemes(family):
    if family == CORE_FAMILY:
        return [(F_PERM_FDR, "permutation"), (F_PARAM_FDR, "parametric")]
    return [(F_PERM_FDR, "permutation")]

# ---- (a) tidy machine-readable CSV across all LNM families / schemes / metrics ----
tidy_rows = []
metrics_by_family = {}   # (family, scheme) -> metrics dict, for the text templates
for fam in LNM_FAMILIES:                              # nocov first, then volnihss
    for pmap, scheme in summary_schemes(fam):
        m = family_scheme_spatial_metrics(fam, pmap)
        if m is None:
            print(f"[skip] {fam} ({scheme}): <2 domains ready.")
            continue
        metrics_by_family[(fam, scheme)] = m
        for metric, (mean, std, n) in m.items():
            tidy_rows.append({"family": fam, "scheme": scheme,
                              "metric": metric, "mean": mean, "std": std, "n": n})

if tidy_rows:
    tidy = pd.DataFrame(tidy_rows, columns=["family", "scheme", "metric", "mean", "std", "n"])
    out_csv = FIG_DIR / "spatial_correlation_summary_allfamilies.csv"
    tidy.to_csv(out_csv, index=False)
    print("saved", out_csv)
    print(tidy.round(4).to_string(index=False))
else:
    print("PALM not finished -- no spatial-correlation summary available.")

In [ ]:
# ---- (b) one human-readable results-section text file per LNM family (PERMUTATION numbers) ----
RESULTS_TEMPLATE = (
    "The permutation-based maps showed low spatial overlap with the connectome's degree map "
    "(mean Dice = {deg_dice_mean:.3f} +/- {deg_dice_std:.3f}), implying that the identified "
    "networks were not driven by nonspecific connectome structure. They also exhibited only "
    "moderate similarity across different cognitive domains (mean Dice = {xdom_dice_mean:.3f} "
    "+/- {xdom_dice_std:.3f}), indicating that the maps did not converge onto a shared, "
    "domain-general pattern."
)

for fam in LNM_FAMILIES:
    m = metrics_by_family.get((fam, "permutation"))
    if m is None:
        print(f"[skip] {fam}: no permutation metrics -- no text file written.")
        continue
    xd_m, xd_s, xd_n = m["cross_domain_dice"]
    xc_m, xc_s, xc_n = m["cross_domain_correlation"]
    dd_m, dd_s, dd_n = m["degree_dice"]
    dc_m, dc_s, dc_n = m["degree_correlation"]

    para = RESULTS_TEMPLATE.format(
        deg_dice_mean=dd_m, deg_dice_std=dd_s,
        xdom_dice_mean=xd_m, xdom_dice_std=xd_s,
    )
    extra = (
        "\n\nSupporting spatial-correlation statistics (permutation scheme):\n"
        f"  - Cross-domain Dice (domain-pairs):                mean = {xd_m:.3f} +/- {xd_s:.3f}  (n={xd_n})\n"
        f"  - Cross-domain spatial correlation (signed T):     mean r = {xc_m:.3f} +/- {xc_s:.3f}  (n={xc_n})\n"
        f"  - Degree Dice (per domain vs degree-95th binary):  mean = {dd_m:.3f} +/- {dd_s:.3f}  (n={dd_n})\n"
        f"  - Degree spatial correlation (signed T vs degree): mean r = {dc_m:.3f} +/- {dc_s:.3f}  (n={dc_n})\n"
    )
    header = (
        f"Family: {FAMILY_PRETTY.get(fam, fam)} ({fam})\n"
        f"Scheme: permutation (TFCE FDR, F_PERM_FDR), threshold -log10(p) >= {float(THR):.4f}\n"
        f"Degree binary: 95th-percentile of {DEGREE_MAP.name}\n"
        + "=" * 78 + "\n\n"
    )
    text = header + para + extra
    out_txt = FIG_DIR / f"spatial_correlation_summary__{fam}.txt"
    out_txt.write_text(text)
    print("saved", out_txt)
    print(text)
    print("-" * 78)